# 第7章 1 枚の画像から未来を動画にする (Image-to-Video)

ここから Cosmos 3 Edge の **Generator** (動画などを作る部分) を使います。最初は一番基本の「画像 → 動画」です。

1. お手本の画像と JSON プロンプトで動画を作る
2. JSON プロンプトの中身を見る (なぜ文章ではなく JSON なのか)
3. 自分の画像と指示文から、Reasoner に JSON プロンプトを書かせる (プロンプトのアップサンプリング)
4. その JSON で動画を作る。seed を変えて何本か作る
5. 文章だけから動画を作る (Text-to-Video)

**Edge で作れる動画**: 256p (320×192 など) か 480p (832×480 など)、12〜30 fps、50〜150 フレーム。縦横比は 16:9 / 4:3 / 1:1 / 3:4 / 9:16。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

## 1. お手本の画像と JSON プロンプトで動画を作る

モデルカードのお手本は 121 フレーム (24fps で約 5 秒) ですが、Colab で早く終わるよう 61 フレーム (約 2.5 秒) にしています。
目安: A100 で数分、L4 ではその数倍かかります。初回はモデルの読み込みにも時間がかかります。

In [ ]:
first_frame = load_image(fetch(f"{HF}/example_i2v_input.jpg"), max_side=10_000)
json_prompt = json.loads(Path(fetch(f"{HF}/example_i2v_prompt.json")).read_text())
negative_prompt = json.loads(Path(fetch(f"{HF}/negative_prompt.json")).read_text())
display(first_frame.resize((416, 240)))

In [ ]:
pipe = load_pipe(flow_shift=12.0)
result = pipe(
    prompt=json.dumps(json_prompt),
    negative_prompt=json.dumps(negative_prompt),
    image=first_frame,
    num_frames=61,
    height=480,
    width=832,
    fps=24.0,
    num_inference_steps=20,
    guidance_scale=6.0,
    generator=torch.Generator(device=DEVICE).manual_seed(0),
    add_resolution_template=False,
    add_duration_template=False,
)
export_to_video(result.video, "ch07_example.mp4", fps=24, macro_block_size=1)
show_video("ch07_example.mp4", 832)

## 2. JSON プロンプトの中身を見る

Cosmos 3 の Generator は、短い文よりも**場面を細かく書いた決まった形の JSON** で学習されています。
被写体・背景・光・カメラの動きに加えて、`actions` や `segments` に**時刻つきで何が起きるか**を書くのが特徴です。
公式では、短い指示文を大きな言語モデルでこの形に書き直す「プロンプトのアップサンプリング」を推奨しています。

In [ ]:
print("キー:", list(json_prompt.keys()), "\n")
for key in ("temporal_caption", "actions", "segments", "camera"):
    if key in json_prompt:
        print(f"[{key}]")
        print(json.dumps(json_prompt[key], indent=2, ensure_ascii=False)[:800], "\n")

## 3. 自分の画像と指示文から、Reasoner に JSON プロンプトを書かせる

公式の例では Claude などの外部の大きなモデルでアップサンプリングしますが、ここでは **Cosmos 3 Edge 自身の Reasoner** に書かせます (API キー不要)。
GPU メモリを空けるため、Generator を一度降ろして Reasoner を読み込み、書き終わったら Reasoner を降ろします。

`use_sample` に印を付けると、アップロードの代わりにお手本のロボットの画像を使います。

In [ ]:
# @title 画像と指示文
import math
from PIL import ImageOps

instruction = "ロボットアームが花をつかんで、赤いボトルに入れる"  # @param {type:"string"}
num_frames = 61  # @param {type:"slider", min:53, max:149, step:4}
use_sample = True  # @param {type:"boolean"}
FPS = 24

if use_sample:
    src = load_image(fetch(f"{HF}/example_reasoning_input.png"), max_side=10_000)
else:
    from google.colab import files
    up = files.upload()
    src = load_image(next(iter(up)), max_side=10_000)

# 480p で対応している縦横比のうち、画像に一番近いものに合わせて中央を切り抜く (幅, 高さ)
SIZES = {"16,9": (832, 480), "4,3": (736, 544), "1,1": (640, 640), "3,4": (544, 736), "9,16": (480, 832)}
aspect = min(SIZES, key=lambda k: abs(math.log(SIZES[k][0] / SIZES[k][1] / (src.width / src.height))))
W, H = SIZES[aspect]
my_frame = ImageOps.fit(src, (W, H), Image.BICUBIC)
my_frame.save("ch07_first_frame.png")
print(f"{W}x{H} ({aspect.replace(',', ':')}), {num_frames} フレーム = {num_frames / FPS:.1f} 秒")
display(my_frame.resize((W // 2, H // 2)))

In [ ]:
# @title Reasoner に JSON プロンプトを書かせる
free_gpu()
load_reasoner()

# お手本の値をそのまま見せると、Edge は背景などを写してしまう。値の代わりに「何を書くか」のひな形を見せ、
# 主なキーだけを書かせてから、お手本と同じキーの形に入れ直す
SCHEMA = {
    "subjects": [{"description": "what the main subject looks like", "location": "where it is in the frame"}],
    "background_setting": "the background that is really in the image",
    "lighting": "lighting conditions",
    "cinematography": "camera angle and camera motion (usually a fixed camera)",
    "actions": [{"time": "0:00-0:01", "description": "what happens"}, {"time": "0:01-0:02", "description": "what happens next"}],
    "segments": [{"time": "0:00-0:02", "description": "summary of this part"}],
    "temporal_caption": "one paragraph that describes the whole video in time order",
}

def to_full_prompt(part, template):
    """書かせた主なキーを、お手本と同じキーの形 (ほかのキーは空) に入れ直す"""
    full = {k: ([] if isinstance(v, list) else {} if isinstance(v, dict) else "") for k, v in template.items()}
    full.update({k: v for k, v in (part or {}).items() if k in full})
    return full

duration = num_frames / FPS
request = f"""You write prompts for an image-to-video model. The attached image is the first frame of the video.
What should happen in the video: {instruction}
(The instruction may be written in Japanese; translate it.)
Video length: {duration:.1f} seconds.

Describe this video as one JSON object with exactly these keys. The values below only explain what to write;
replace every value with a description of THIS image and THIS instruction.
{json.dumps(SCHEMA, ensure_ascii=False)}
Describe only what is really in the image. Use timestamps within the video length.
Write every value in English. Output only the JSON."""

answer = ask(request, "ch07_first_frame.png", max_new_tokens=3072)
part = parse_json(answer)
my_prompt = to_full_prompt(part, json_prompt) if isinstance(part, dict) else None
if not isinstance(my_prompt, dict):
    # 崩れた JSON が返ってきたら、指示文だけを入れた最小の形で代用する
    print("JSON として読めなかったので、指示文だけで組みます\n", answer[:500])
    my_prompt = {k: ([] if isinstance(v, list) else {} if isinstance(v, dict) else "") for k, v in json_prompt.items()}
    my_prompt["temporal_caption"] = instruction
    my_prompt["actions"] = [{"time": f"0:00-0:{math.ceil(duration):02d}", "description": instruction}]

# 解像度と長さは実際に作る値に揃える
my_prompt.update({"resolution": {"W": W, "H": H}, "aspect_ratio": aspect, "duration": f"{round(duration)}s", "fps": FPS})
Path("ch07_my_prompt.json").write_text(json.dumps(my_prompt, indent=2, ensure_ascii=False))
print(json.dumps(my_prompt, indent=2, ensure_ascii=False)[:2500])
free_gpu()

JSON を手で直したいときは、左のファイル欄で `ch07_my_prompt.json` をダブルクリックして編集・保存してから次のセルを実行してください (次のセルはこのファイルを読み直します)。
動きが弱いときは `actions` と `segments` の説明を具体的にすると効きます。

## 4. その JSON で動画を作る

同じ JSON でも seed (乱数の種) で出来がかなり変わります。`n_videos` を増やすと seed を 1 つずつずらして何本か作ります。

In [ ]:
# @title 動画を作る
seed = 0  # @param {type:"integer"}
n_videos = 2  # @param {type:"slider", min:1, max:4, step:1}

pipe = load_pipe(flow_shift=12.0)
my_prompt = json.loads(Path("ch07_my_prompt.json").read_text())
for s in range(seed, seed + n_videos):
    result = pipe(
        prompt=json.dumps(my_prompt),
        negative_prompt=json.dumps(negative_prompt),
        image=Image.open("ch07_first_frame.png"),
        num_frames=num_frames,
        height=H,
        width=W,
        fps=float(FPS),
        num_inference_steps=20,
        guidance_scale=6.0,
        generator=torch.Generator(device=DEVICE).manual_seed(s),
        add_resolution_template=False,
        add_duration_template=False,
    )
    out = f"ch07_my_video_seed{s}.mp4"
    export_to_video(result.video, out, fps=FPS, macro_block_size=1)
    print(out)
    show_video(out, min(W, 832))

## 5. 文章だけから動画を作る (Text-to-Video)

画像を渡さなければ、JSON プロンプトだけから動画を作ります。お手本の JSON の場面を、最初のコマなしで作らせてみます。
画像という手がかりがない分、場面の作り込みは JSON の書き方次第です。

In [ ]:
result = pipe(
    prompt=json.dumps(json_prompt),
    negative_prompt=json.dumps(negative_prompt),
    num_frames=61,
    height=480,
    width=832,
    fps=24.0,
    num_inference_steps=20,
    guidance_scale=6.0,
    generator=torch.Generator(device=DEVICE).manual_seed(0),
    add_resolution_template=False,
    add_duration_template=False,
)
export_to_video(result.video, "ch07_t2v.mp4", fps=24, macro_block_size=1)
show_video("ch07_t2v.mp4", 832)

## うまくいかないとき

- **ほとんど動かない**: JSON の `actions` と `segments` に、誰が・何を・どちらへ動かすかを時刻つきで具体的に書きます
- **形が崩れる・別の物に変わる**: 長い動画ほど崩れやすいので `num_frames` を減らすか、seed を変えて何本か作ります
- **速く試したい**: `height=256` 系 (16:9 なら 320×192) にすると数倍速くなります (画質は落ちます)
- **真っ黒・空の動画になる**: 安全フィルタに止められた可能性があります。画像や seed を変えて試してください
- 作られるのは「それらしい未来」で、物理シミュレーションの結果ではありません